In [2]:
#reset all variables
%reset -f

In [3]:
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, RadioMapSolver, ITURadioMaterial, PathSolver
import mitsuba as mi
import numpy as np
import pandas as pd
import drjit as dr
dr.set_flag(dr.JitFlag.Debug, True)

jitc_llvm_init(): LLVM API initialization failed ..


In [4]:
scene = load_scene("scenev3/final_scene_v3.xml",merge_shapes=False)
scene.frequency = 29.5e9 #Hz
scene.synthetic_array = True

tx_power_dbm = 31

#Create material and thickness
matAndThick = ITURadioMaterial(name="plasterboard",
                               itu_type="plasterboard",
                               thickness=0.09)#m

#See objects and materials from blender
for name, obj in scene.objects.items():
    print(f'\033[1m{name:}---{obj.radio_material.name}\033[0m')
    if obj.radio_material.name == "itu_plasterboard":
        obj.radio_material = matAndThick #Applies thickness
        print("    -Thickness applied")

Plane---itu_concrete
C1-1---itu_plasterboard
    -Thickness applied
C1-2---itu_plasterboard
    -Thickness applied
C1-3---itu_plasterboard
    -Thickness applied
C1-4---itu_plasterboard
    -Thickness applied
C1-5---itu_plasterboard
    -Thickness applied
C1-6---itu_plasterboard
    -Thickness applied
C1-7---itu_plasterboard
    -Thickness applied
C1-8---itu_plasterboard
    -Thickness applied
C1-9---itu_plasterboard
    -Thickness applied
C2-1---itu_plasterboard
    -Thickness applied
C2-2---itu_plasterboard
    -Thickness applied
C2-3---itu_plasterboard
    -Thickness applied
C2-4---itu_plasterboard
    -Thickness applied
C2-5---itu_plasterboard
    -Thickness applied
C2-6---itu_plasterboard
    -Thickness applied
C2-7---itu_plasterboard
    -Thickness applied
C2-8---itu_plasterboard
    -Thickness applied
C2-9---itu_plasterboard
    -Thickness applied
C2-10---itu_plasterboard
    -Thickness applied
C2-11---itu_plasterboard
    -Thickness applied
C2-12---itu_plasterboard
    -Thickne

In [5]:
# Configure antennas arrays
scene.tx_array = PlanarArray( #Transmitter
    num_rows=1,
    num_cols=4,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="iso",
    polarization = "V")

scene.rx_array = PlanarArray( #Receiver
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="dipole",
    polarization="V")

#Create transmitter and receiver
tx = Transmitter(name="tx",
                 position=mi.Point3f(-7, 0, 1),
                 orientation=mi.Point3f(0, 0, 0),
                 power_dbm= tx_power_dbm)
scene.add(tx)
tx.look_at([-6, 0, 2])
numreceptors = ["1", "2", "3", "4", "5", "6", "7", "7b", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "17b", "18", "19", "20"]
nReceptor = 22
positions = ( (-6.5 , 4),   #1
              (-4   , 3),   #2
              (-2   , 3),   #3
              (0    , 3),   #4
              (1    , 1.2), #5
              (-2   , 1.2), #6
              (-6   , 1.2), #7
              (-4.5 , 1.2), #7b
              (-5.3 , -0.5),#8
              (-2.5 , 0.2), #9
              (0.5  , 0.5), #10
              (2.5  , 1),   #11
              (3    , -1),  #12
              (1    , -2),  #13
              (-2   , -2),  #14
              (-5.3 , -2.5),#15
              (-7   , -2.5),#16
              (-5.7 , -4),  #17
              (-4   , -4),  #17b
              (-2   , -4),  #18
              (1    , -4),  #19
              (3    , -4))  #20
for i in range(nReceptor):
    posx = positions[i][0]
    posy = positions[i][1]
    posz = 1
    rx = Receiver(name=f"rx_{i}",
                 position=mi.Point3f(posx, posy, posz),
                 orientation=mi.Point3f(0.0, 0.0, 0.0))
    scene.add(rx)

In [ ]:
# Radio Map
rmSolver = RadioMapSolver()
rm = rmSolver(scene,
              samples_per_tx= int(1e8),
              max_depth=6,
              cell_size=(0.1, 0.1),
              los = True,
              specular_reflection = True,
              diffuse_reflection = True,
              refraction = True,
              diffraction = True,
              edge_diffraction = True,
              diffraction_lit_region = True,
              stop_threshold = -131)

scene.preview(radio_map=rm, clip_at=3)
rm.show(metric='path_gain', vmin=-131)
rm.show(metric='rss', vmin=-100, vmax=-40)


Dr.Jit encountered an unrecoverable error and will now shut
down. Please re-run your program in debug mode to check for
out-of-bounds reads, writes, and other sources of undefined
behavior. You can do so by calling

   dr.set_flag(dr.JitFlag.Debug, True)

at the beginning of the program. If these additional checks
fail to pinpoint the problem, then you have likely found a
bug. We are happy to help investigate and fix the problem if
you can you create a self-contained reproducer and submit it
at https://github.com/mitsuba-renderer/drjit.

The error message of this specific failure is as follows:
>>> jit_optix_compile(): optixModuleGetCompilationState() indicates that the compilation did not complete succesfully. The module's compilation state is: 0x2363
Please see the PTX assembly listing and error message below:

.version 9.0
.target sm_75
.address_size 64

.const .align 8 .b8 params[896];

.entry __raygen__8ed6b71a275e71c398fa86f24b12ac87() {
    .reg.b8  %b <2125>; .reg.b16  %w<2125

In [ ]:
#Paths
solver  = PathSolver()
paths = solver(scene,
               max_depth=6,
               max_num_paths_per_src= int(1e6),
               samples_per_src= int(1e6),
               synthetic_array=True,
               los=True,
               specular_reflection=True,
               diffuse_reflection=True,
               refraction=True,
               edge_diffraction=True,
               diffraction_lit_region=True)

#Ampltiudes and TDoA from CIR
a, tau = paths.cir(out_type="numpy")

# print("rx, rx_ant, tx, tx_ant, paths, batch")
# print(a.shape)
# print("rxs, txs, paths")
# print(tau.shape)

SimPowerdB = np.zeros(22)

#print("\n--- RSS ---")

for rx in range(nReceptor):
    amp = a[rx, 0, 0, :, :, 0]
    power = np.sum(np.abs(amp)**2)
    dBPower = tx_power_dbm + 10*np.log10(power + 1e-15)
    SimPowerdB[rx] = dBPower

    #print(f"\nReceptor {numreceptors[rx]} ubicado en {positions[rx]} tiene RSS {dBPower:.2f} dBm")

In [ ]:
df = pd.read_csv("Resultados.csv")
data = df["90"].to_numpy()

for i in range(nReceptor):
    d1 = data[i]
    d2 = SimPowerdB[i]
    r = np.abs(d1) - np.abs(d2)
    print(f"La diferencia en el receptor {numreceptors[i]} es: {r:.2f} dBm" )

